In [2]:
import sys
sys.path.append("../src")

from ch14_full_adder import *

In [14]:
def int_to_nbit_list(num, nbits=8):
    """
    Converts an integer to an n-bit list using our simulated hardware 
    for Two's Complement (Invert and Add 1).
    """
    abs_val = abs(num)
    bin_str = format(abs_val, f'0{nbits}b')
    bit_list = [int(bit) for bit in bin_str]

    if num >= 0:
        return bit_list

    # --- THE HARDWARE WAY ---
    
    # Step A: Pass the bits through our simulated Inverters
    my_inverter = Inverter()
    inverted_bits = [my_inverter([bit]) for bit in bit_list]

    # Step B: Wire up an adder to add 1
    adder = NBitAdderWithCarryOut(nbits)
    
    # Step B: Add 0, but set the Carry-In pin to 1
    zero_bits = [0 for _ in range(nbits)]
    overflow_carry, final_bits = adder(inverted_bits, zero_bits, last_carry_in=1)
    
    return final_bits
    
# Now you can easily create your 8-bit and 16-bit specific helpers:
def int_to_8bit_list(num):
    return int_to_nbit_list(num, nbits=8)

def int_to_16bit_list(num):
    return int_to_nbit_list(num, nbits=16)

In [15]:
def bit_list_to_int(bit_list, signed=True):
    """Converts a bit list (MSB at index 0) back to a standard integer."""
    bin_str = "".join(str(bit) for bit in bit_list)
    val = int(bin_str, 2)
    
    # If we are treating this as a signed Two's Complement number...
    if signed:
        # Check the Most Significant Bit (Index 0)
        msb = bit_list[0]
        if msb == 1:
            # If MSB is 1, it's negative. We calculate its true negative value.
            # E.g., for 8 bits: val - 2^8 (val - 256)
            val = val - (1 << len(bit_list)) 
            
    return val

In [16]:
# Initialize 8-bit adder
adder8 = NBitAdderWithCarryOut(8)

def run_test_adder8(test_name, num1, num2):
    print(f"--- {test_name} ---")
    
    # Convert inputs to 16-bit hardware format
    x_bits = int_to_8bit_list(num1)
    y_bits = int_to_8bit_list(num2)

    # Run the hardware forward pass
    final_carry, sum_bits = adder8(x_bits, y_bits)
    # Convert result back to human-readable integer
    result_int = bit_list_to_int(sum_bits)

    # Display results
    print(f"  {num1} in binary: {x_bits}")
    print(f"+ {num2} in binary: {y_bits}")
    print("-" * 65)
    print(f"  Result bits:    {sum_bits}")
    print(f"  Final Carry Out: {final_carry}")
    print(f"  Decimal Result:  {result_int}")
    print(f"  Expected Sum:    {(num1 + num2) % 256}\n")
    print(f"  Expected Carry Out: {(num1 + num2) // 256}\n")

# ==========================================
# TEST CASES: STANDARD ADDITION
# ==========================================

# Test 1:
run_test_adder8("Test 1: Simple Addition", -1, 25)

# Test 1:
run_test_adder8("Test 1: Simple Addition", -1, -1)

--- Test 1: Simple Addition ---
  -1 in binary: [1, 1, 1, 1, 1, 1, 1, 1]
+ 25 in binary: [0, 0, 0, 1, 1, 0, 0, 1]
-----------------------------------------------------------------
  Result bits:    [0, 0, 0, 1, 1, 0, 0, 0]
  Final Carry Out: 1
  Decimal Result:  24
  Expected Sum:    24

  Expected Carry Out: 0

--- Test 1: Simple Addition ---
  -1 in binary: [1, 1, 1, 1, 1, 1, 1, 1]
+ -1 in binary: [1, 1, 1, 1, 1, 1, 1, 1]
-----------------------------------------------------------------
  Result bits:    [1, 1, 1, 1, 1, 1, 1, 0]
  Final Carry Out: 1
  Decimal Result:  -2
  Expected Sum:    254

  Expected Carry Out: -1



In [17]:
bit_s = bit_list_to_int([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ])
print(bit_s)

-1


In [ ]:
class NBitAdderWithOverflow:
    def __init__(self, nbits):
        self.nbits = nbits # Carry_In, A, and B
        # Instantiate 'n' Full Adders
        self.fulladders = [FullAdder() for _ in range(self.nbits)]

        # below gate use for overflow
        self.inverter = Inverter()
        # AND gate detects an overflow condition for negative numbers.
        # the sign bit of x and y inputs are both 1 and the sign
        # of the sum is 0.
        self.and_gate = AND(nin=3)
        # NOR gate detects an overflow condition for positive numbers.
        # the sign bit of x and y inputs are both 0 and the sign
        # of the sum is 1.
        self.nor_gate = NOR(nin=3)
        
        self.or_gate = OR(nin=2)
        
    def __call__(self, x, y, last_carry_in=0):
        assert len(x) == self.nbits and len(y) == self.nbits, f"Inputs must be {self.nbits} bits long"
        final_sums = []
        final_carrys = [last_carry_in]

        # Iterate backwards: start from the far right and move left
        for idx in reversed(range(self.nbits)):
            carry_in = final_carrys[0]
            a = x[idx]
            b = y[idx]
            
            # Run the Full Adder for this column
            carry_out, sum_out = self.fulladders[idx]([carry_in, a, b])
            
            final_carrys = [carry_out] + final_carrys
            final_sums = [sum_out] + final_sums
        # The final carry out of the MSB is still sitting at index 0

        # we return third result that reflect overflow use two-complement
        invert_msb_of_final_sum = self.inverter([final_sums[0]])
        msb_of_x = x[0]
        msb_of_y = y[0]

        out_of_and_gate = self.and_gate([invert_msb_of_final_sum, msb_of_x, msb_of_y])
        out_of_nor_gate = self.nor_gate([invert_msb_of_final_sum, msb_of_x, msb_of_y])
        overflow = self.or_gate([out_of_and_gate, out_of_nor_gate])
        
        return overflow, final_carrys[0], final_sums

In [ ]:
# Initialize our new upgraded 8-bit adder
adder8_tc = NBitAdderWithOverflow(8)

def run_overflow_test(test_name, num1, num2):
    print(f"--- {test_name} ---")
    
    # 1. Convert human numbers to 8-bit hardware lists
    x_bits = int_to_8bit_list(num1)
    y_bits = int_to_8bit_list(num2)
    
    # 2. Run the hardware forward pass
    overflow_flag, final_carry, sum_bits = adder8_tc(x_bits, y_bits)
    
    # 3. Read the result back into a human number
    result_int = bit_list_to_int(sum_bits, signed=True)
    
    print(f"  {num1:4} in binary: {x_bits}")
    print(f"+ {num2:4} in binary: {y_bits}")
    print("-" * 65)
    print(f"  Result bits:       {sum_bits}")
    print(f"  Decimal Result:    {result_int}")
    print(f"  Hardware Overflow? {'YES (OVERFLOW!)' if overflow_flag else 'No (Safe)'}")
    print("\n")

# ==========================================
# TEST CASES: 8-BIT TWO'S COMPLEMENT LIMITS (-128 to +127)
# ==========================================

# Test 1: Safe Positive Addition 
# 50 + 50 = 100 (Within the +127 limit)
run_overflow_test("Test 1: Safe Positive Addition", 50, 50)

# Test 2: Safe Negative Addition
# -50 + -50 = -100 (Within the -128 limit)
run_overflow_test("Test 2: Safe Negative Addition", -50, -50)

# Test 3: Mixed Addition (Impossible to overflow)
# 120 + (-120) = 0
run_overflow_test("Test 3: Mixed Signs (Never Overflows)", 120, -120)

# Test 4: Positive Overflow!
# 100 + 50 = 150. This exceeds +127. 
# The bits will wrap around into negative territory, and the flag should trigger.
run_overflow_test("Test 4: POSITIVE OVERFLOW", 100, 50)

# Test 5: Negative Overflow!
# -100 + -50 = -150. This exceeds -128.
# The bits will wrap around into positive territory, and the flag should trigger.
run_overflow_test("Test 5: NEGATIVE OVERFLOW", -100, -50)

--- Test 1: Safe Positive Addition ---
    50 in binary: [0, 0, 1, 1, 0, 0, 1, 0]
+   50 in binary: [0, 0, 1, 1, 0, 0, 1, 0]
-----------------------------------------------------------------
  Result bits:       [0, 1, 1, 0, 0, 1, 0, 0]
  Decimal Result:    100
  Hardware Overflow? No (Safe)


--- Test 2: Safe Negative Addition ---
   -50 in binary: [1, 1, 0, 0, 1, 1, 1, 0]
+  -50 in binary: [1, 1, 0, 0, 1, 1, 1, 0]
-----------------------------------------------------------------
  Result bits:       [1, 0, 0, 1, 1, 1, 0, 0]
  Decimal Result:    -100
  Hardware Overflow? No (Safe)


--- Test 3: Mixed Signs (Never Overflows) ---
   120 in binary: [0, 1, 1, 1, 1, 0, 0, 0]
+ -120 in binary: [1, 0, 0, 0, 1, 0, 0, 0]
-----------------------------------------------------------------
  Result bits:       [0, 0, 0, 0, 0, 0, 0, 0]
  Decimal Result:    0
  Hardware Overflow? No (Safe)


--- Test 4: POSITIVE OVERFLOW ---
   100 in binary: [0, 1, 1, 0, 0, 1, 0, 0]
+   50 in binary: [0, 0, 1, 1, 0